# Export to phy with and without recording

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.postprocessing as spost
import spikeinterface.exporters as sexp

import numpy as np
import shutil

from pathlib import Path

%matplotlib widget

In [8]:
data_folder = Path(r'F:\acuteBehavior\707535\707535_2024-04-19_10-01-31')
phy_folder = Path(data_folder, 'forPhy')

In [4]:
raw_compressed_folder = data_folder / "Record Node 104" 
postprocessed_folder = data_folder / "sorted" / "postprocessed"

In [ ]:
recording = 

In [5]:
stream_names =  [p.name for p in postprocessed_folder.iterdir() if p.is_dir()]

In [6]:
# we do it for 1 stream first
stream_name = stream_names[0]

### Load waveforms in recordingless mode

In [7]:
we_recless = si.load_waveforms(postprocessed_folder / stream_name, with_recording=False)

/root/capsule/code/spikeinterface/src/spikeinterface/core/base.py:998: UserWarning: Versions are not the same. This might lead compatibility errors. Using spikeinterface==0.97.1 is recommended
  warnings.warn(


The `WaveformExtractor` contains all relevant metadata of the recording, except traces

In [8]:
we_recless.channel_ids[:5]

array(['AP1', 'AP2', 'AP3', 'AP4', 'AP5'], dtype='<U5')

In [9]:
len(we_recless.channel_ids)

350

In [10]:
we_recless.get_channel_locations()[:5]

array([[ 0.,  0.],
       [48.,  0.],
       [ 0., 20.],
       [48., 20.],
       [ 0., 40.]])

We need a trick here because the export to phy attempts to add the template similarity to a read-only file. We can fake that the object is in-memory only.

In [12]:
we_recless.get_available_extension_names()

['template_metrics',
 'similarity',
 'principal_components',
 'spike_amplitudes',
 'correlograms',
 'isi_histograms',
 'spike_locations',
 'unit_locations',
 'quality_metrics']

### Export to Phy without recording

In [13]:
phy_folder = results_folder / f"{postprocessed_folder.parent.name}_phy"

In [14]:
we_recless.is_read_only()

True

In [15]:
sexp.export_to_phy(we_recless, 
                   output_folder=phy_folder,
                   compute_pc_features=False,
                   remove_if_exists=True,
                   copy_binary=False)

Run:
phy template-gui  /root/capsule/results/ecephys_625464_2022-10-06_11-22-45_sorted-ks25_phy/params.py


In [18]:
spike_locations = we_recless.load_extension("spike_locations").get_data()
spike_depths = spike_locations["y"]

In [19]:
np.save(phy_folder / "spike.depths.npy", spike_depths)

### Export to Phy with recording

We first need to load the associated recording

In [20]:
compressed_file_name = stream_name[:stream_name.find("_recording")]

In [21]:
compressed_file_name

'experiment1_Record Node 104#Neuropix-PXI-100.ProbeA-AP'

In [22]:
print(raw_compressed_folder / f"{compressed_file_name}.zarr")

data/ecephys_625464_2022-10-06_11-22-45/ecephys_compressed/experiment1_Record Node 104#Neuropix-PXI-100.ProbeA-AP.zarr


In [23]:
recording = si.read_zarr(raw_compressed_folder / f"{compressed_file_name}.zarr")

We now need to filter out the bad channels:

In [25]:
good_channel_mask = np.in1d(recording.channel_ids, we_recless.channel_ids)
recording_good = recording.channel_slice(recording.channel_ids[good_channel_mask])
recording_good

ChannelSliceRecording: 350 channels - 30.0kHz - 1 segments - 56,439,756 samples 
                       1,881.33s (31.36 minutes) - int16 dtype - 36.79 GiB

Now we can set the recording to the recordingless waveforms (maybe we hp filter first!):

In [26]:
recording_hp = spre.highpass_filter(recording_good)

In [27]:
we_recless.set_recording(recording_hp)

In [29]:
we_recless.recording

HighpassFilterRecording: 350 channels - 30.0kHz - 1 segments - 56,439,756 samples 
                         1,881.33s (31.36 minutes) - int16 dtype - 36.79 GiB

In [31]:
phy_raw_folder = results_folder / f"{postprocessed_folder.parent.name}_phy_raw"

In [32]:
sexp.export_to_phy(we_recless, 
                   output_folder=phy_raw_folder,
                   compute_pc_features=False,
                   remove_if_exists=True,
                   copy_binary=True,
                   n_jobs=8,
                   progress_bar=True,
                   chunk_duration="1s")

write_binary_recording:   0%|          | 0/1882 [00:00<?, ?it/s]

Run:
phy template-gui  /root/capsule/results/ecephys_625464_2022-10-06_11-22-45_sorted-ks25_phy_raw/params.py


In [33]:
np.save(phy_raw_folder / "spike.depths.npy", spike_depths)